<a href="https://colab.research.google.com/github/nakkaramyamanasa/datascience_masters/blob/branch_v/custom_guardrails_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Define exact folders from the Git repo
folders = [
    "custom-guardrail/configs/data_generation/synthetic_pii",
    "custom-guardrail/configs/data_generation/synthetic_injection",
    "custom-guardrail/configs/data_generation/gold_dataset",
    "custom-guardrail/src/utils",
    "custom-guardrail/src/data_generation/hf_collection",
    "custom-guardrail/src/finetuning/domain_data",
    "custom-guardrail/src/finetuning/domain_finetuning",
    "custom-guardrail/src/model_training/model_adaptation",
    "custom-guardrail/src/model_evaluation/gold_evaluation",
    "custom-guardrail/src/model_evaluation/metrics",
    "custom-guardrail/src/inference",
    "custom-guardrail/src/traceability",
    "custom-guardrail/experimental",
    "custom-guardrail/tests/unit",
    "custom-guardrail/tests/fixtures",
    "custom-guardrail/docs/adr"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Create empty __init__.py files so Python treats directories as packages for imports
for root, dirs, _ in os.walk("custom-guardrail/src"):
    for d in dirs:
        init_file = os.path.join(root, d, "__init__.py")
        if not os.path.exists(init_file):
            open(init_file, 'a').close()

print("✅ Repository directory structure matched 100% to production screenshots!")

✅ Repository directory structure matched 100% to production screenshots!


In [ ]:
%%writefile custom-guardrail/configs/pipeline_config.py

import torch

SEED = 42 # What is seed?
MAX_LENGTH = 128 # What is max length
MODEL_ID = "microsoft/deberta-v3-small"

ID2LABEL = {0: "SAFE", 1: "POSITIVE"}
LABEL2ID = {"SAFE": 0, "POSITIVE": 1}

TASK_CFG = {
    "pii": {
        "train_path": "custom-guardrail/outputs/pii/train.jsonl",
        "val_path": "custom-guardrail/outputs/pii/val.jsonl",
        "test_path": "custom-guardrail/outputs/pii/test.jsonl",
        "gold_path": "custom-guardrail/outputs/pii/gold.jsonl",
    }
}

# Training hyperparameters with rationale for meetings
TRAIN_HP = {
    "head" : {
        # High LR: Trains only the random classification head (~2M params) while backbone is fronzen
        "learning_rate": 1e-3,
        "batch_size": 64,
        "epochs": 3,
        "fp16": False # Forced fp32 to prevent DeBERTa gradient NaN issues.
    },
    "full": {
        # Low LR: Gently updates all 141 parameters witout catastrophic forgetting.
        "learning_rate": 2e-5,
        "batch_size": 32,
        "epochs": 5,
        "fp16": False
    }
}

Overwriting custom-guardrail/configs/pipeline_config.py


In [ ]:
%%writefile custom-guardrail/src/data_generation/validators.py


import re

def validate_pii_sample(text: str, label: int, categories: list) -> tuple[bool, str]:
  """
  Enforce content validation rules
  - Positive sample contains detectable pattern for claimed subtype.
  - Negative sample does not contain detectable positive patterns.
  """

  email_regex = r'[\w\.-]+@[\w\.-]+\.\w+'
  phone_regex = r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b'
  ssn_regex = r'\b\d{3}-\d{2}-\d{4}\b'

  if label == 0:
    # Conflict check: SAFE samples MUST NOT acidentally contain PII
    if re.search(email_regex, text) or re.search(phone_regex, text) or re.search(ssn_regex, text):
      return False, "Conflict: SAFE sample contains PII pattern"
    return True, "Valid SAFE sample"

  elif label == 1:
    # Coverage check: POSITIVE sample MUST contain the pattern it claims to have
    if "EMAIL" in categories and not re.search(email_regex, text):
      return False, "Missing EMAIL pattern"
    if "PHONE" in categories and not re.search(phone_regex, text):
      return False, "Missing PHONE pattern"
    if "SSN" in categories and not re.search(ssn_regex, text):
      return False, "Missing SSN pattern"
    return True, "Valid POSITIVE sample"

  return False, "Invalid label"

Overwriting custom-guardrail/src/data_generation/validators.py


In [ ]:
%%writefile custom-guardrail/src/data_generation/generate.py

import json
import os
import sys
import random
import torch
from transformers import pipeline

# Add the current directory sys.path so we can import the validator
sys.path.append(os.path.dirname(__file__))
from validators import validate_pii_sample

def load_llm():
    """Loads a small open-source LLM for local generation"""
    print("Loading local LLM Qwen2.5-0.5B-Instruct for synthetic generation")
    device = 0 if torch.cuda.is_available() else -1
    generator = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-0.5B-Instruct",
        device=device,
        torch_dtype=torch.bfloat16 if device == 0 else torch.float32
    )
    print("LLM loaded successfully.")
    return generator

def generate_phase1_dataset():
    llm = load_llm()
    random.seed(42)

    # In production, these prompts are loaded from configs/data_generation/synthetic_pii/pii_generation.yaml
    generation_tasks = [
        {"prompt": "Write a random technical question about Kubernetes.", "label": 0, "categories": []},
        {"prompt": "Write a question about Python pandas.", "label": 0, "categories": []},
        {"prompt": "Write a customer support request that contains a fake email address.", "label": 1, "categories": ["EMAIL"]},
        {"prompt": "Write a message asking to update a phone number, including a fake 10-digit number.", "label": 1, "categories": ["PHONE"]},
        {"prompt": "Write a message stating a fake SSN formatted as XXX-XX-XXXX.", "label": 1, "categories": ["SSN"]}
    ]

    raw_samples = []
    target_samples = 300 # Keep it small for the PoC speed

    print(f"Generating {target_samples} validated samples using LLM...")

    attempts = 0
    while len(raw_samples) < target_samples and attempts < 150:
        attempts += 1
        task = random.choice(generation_tasks)

        # Format prompt for the LLM using the chat template
        messages = [
            {"role": "system", "content": "You are a data generator. Output ONLY the raw requested text. No filler."},
            {"role": "user", "content": task["prompt"]}
        ]
        # tokenize=False returns a string
        prompt_text = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate text - temperature must be > 0 if do_sample=True, or use do_sample=False for greedy
        output = llm(prompt_text, max_new_tokens=50, do_sample=False, pad_token_id=llm.tokenizer.eos_token_id)
        generated_text = output[0]['generated_text'].split("<|im_start|>assistant\n")[-1].strip()

        # Pass the LLM output through validators
        is_valid, reason = validate_pii_sample(generated_text, task["label"], task["categories"])
        if is_valid:
            raw_samples.append({
                "text": generated_text,
                "label": task["label"],
                "pii_categories": task["categories"],
                "source": "Qwen2.5-0.5B-Instruct"
            })
            sys.stdout.write(f"\r✅ Validated samples: {len(raw_samples)}/{target_samples}")
            sys.stdout.flush()

    print("\n\n📊 Splitting dataset into Train/Val/Test/Gold...")

    total = len(raw_samples)
    train_end = int(total * 0.6)
    val_end = train_end + int(total * 0.2)
    test_end = val_end + int(total * 0.1)

    splits = {
        "train": raw_samples[:train_end],
        "val": raw_samples[train_end:val_end],
        "test": raw_samples[val_end:test_end],
        "gold": raw_samples[test_end:]
    }

    out_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../outputs/pii"))
    os.makedirs(out_dir, exist_ok=True)

    for split_name, records in splits.items():
        file_path = os.path.join(out_dir, f"{split_name}.jsonl")
        with open(file_path, "w") as f:
            for r in records:
                r["split"] = split_name
                f.write(json.dumps(r) + "\n")
        print(f"📄 Written {len(records)} records to outputs/pii/{split_name}.jsonl")

if __name__ == "__main__":
    generate_phase1_dataset()

Overwriting custom-guardrail/src/data_generation/generate.py


In [ ]:
!python custom-guardrail/src/data_generation/generate.py

Loading local LLM Qwen2.5-0.5B-Instruct for synthetic generation
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 8075.77it/s]
LLM loaded successfully.
Generating 5000 validated samples using LLM...
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is de

In [ ]:
%%writefile custom-guardrail/src/model_training/model_adaptation/core_logic_ma.py

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import sys
import os

# Add configs directory to path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../configs")))
import pipeline_config as cfg

def get_model_and_tokenizer():
  """Loads the pre-trained backbone and initialises the binary classification head."""
  print(f"Loading tokenizer and model: {cfg.MODEL_ID}...")

  # use_fast=True is requried for DeBERTa tokenizers
  tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_ID, use_fast=True)

  # Load model with exactly 2 labels (0: SAFE, 1: POSTIVE)
  model = AutoModelForSequenceClassification.from_pretrained(
      cfg.MODEL_ID,
      num_labels=2,
      id2label=cfg.ID2LABEL,
      label2id=cfg.LABEL2ID
      )
  return model, tokenizer

def freeze_backbone(model):
  """
  Stage 1: Freezes the 141M backbone parameters.
  Only the ~2M parameters in the 'classifier' and 'pooler' layers will be updated.
  """
  print("Freezing DeBERTa backbone for Stage 1 (Head only)...")
  for name, param in model.deberta.named_parameters():
    # If the parameter is part of the classifiction head, keep it trainable
    if "classifier" in name or "pooler" in name:
      param.requires_grad = True
    else:
      param.requires_grad = False

  # Calculate and print exactly how many parameters are trainable
  trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  print(f"Trainable parameters: {trainable_params}")
  return model

def unfreeze_backbone(model):
  """
  Stage 2: Unfreezes all parameters for the full 141M fine-tune.
  """
  print("🔥 Unfreezing DeBERTa backbone for Stage 2 (Full Fine-Tune)...")
  for param in model.parameters():
      param.requires_grad = True

  trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  print(f"Trainable parameters: {trainable_params:,}")
  return model


In [ ]:
!python custom-guardrail/src/model_training/model_adaptation/core_logic_ma.py

In [ ]:
%%writefile custom-guardrail/src/finetuning/domain_data/core_logic_gd.py

import json
from datasets import Dataset

def load_jsonl(file_path):
  """Parse JSONL files into Hugging Face Dataset Objects."""
  with open(file_path, "r") as f:
    data = [json.loads(line) for line in f]

  # We extract just the text and the label for training
  texts = [d["text"] for d in data]
  labels = [d["label"] for d in data]
  return Dataset.from_dict({"text": texts, "label": labels})

def get_tokenized_datasets(tokenizer, train_path, val_path, max_length):
  """
  Terminology check:
  - Tokenization: Converting words into numerical IDs from the model's vocabulary.
  - Padding: Adding zeros so all sentences in a batch are the exact same length.
  - Truncation: Chopping off words if the sentence exceeds MAX_LENGTH.
  """
  print(f"Loading datasets from {train_path}...")
  train_ds = load_jsonl(train_path)
  val_ds = load_jsonl(val_path)

  def tokenize_batch(batch):
      return tokenizer(
          batch["text"],
          padding="max_length",
          truncation=True,
          max_length=max_length
      )

  print("Tokenizing datasets...")
  # batched=True processes multiple sentences at once, speeding up the CPU workload
  train_tok = train_ds.map(tokenize_batch, batched=True)
  val_tok = val_ds.map(tokenize_batch, batched=True)

  # Convert lists to PyTorch tensors so the GPU can process them
  train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
  val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

  return train_tok, val_tok

Overwriting custom-guardrail/src/finetuning/domain_data/core_logic_gd.py


In [ ]:
%%writefile custom-guardrail/src/finetuning/domain_finetuning/main.py
import sys
import os
import numpy as np
from transformers import Trainer, TrainingArguments
import evaluate

# Setup relative paths
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../configs")))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../model_training/model_adaptation")))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../domain_data")))

import pipeline_config as cfg
from core_logic_ma import get_model_and_tokenizer, freeze_backbone, unfreeze_backbone
from core_logic_gd import get_tokenized_datasets

# Terminology: F1 Score is the harmonic mean of Precision and Recall.
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels, average="binary")

def run_pipeline():
  # 1. Load Model & Tokenizer
  model, tokenizer = get_model_and_tokenizer()

  # 2. Prepare Data
  train_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/train.jsonl"))
  val_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/val.jsonl"))

  train_ds, val_ds = get_tokenized_datasets(tokenizer, train_path, val_path, cfg.MAX_LENGTH)

  # Scaled down for Colab PoC
  batch_size = 8

  # ==========================================
  # STAGE 1: HEAD-ONLY FINE-TUNING
  # ==========================================
  print("\n" + "="*50)
  print("🚀 STAGE 1: Head-Only Fine-Tuning")
  print("="*50)
  model = freeze_backbone(model)

  args_stage1 = TrainingArguments(
      output_dir="custom-guardrail/outputs/checkpoints/stage1",
      learning_rate=cfg.TRAIN_HP["head"]["learning_rate"],
      per_device_train_batch_size=batch_size,
      per_device_eval_batch_size=batch_size,
      num_train_epochs=cfg.TRAIN_HP["head"]["epochs"],
      eval_strategy="epoch",
      save_strategy="epoch",
      fp16=cfg.TRAIN_HP["head"]["fp16"],
      load_best_model_at_end=True,
      report_to="none"
  )

  trainer_stage1 = Trainer(
      model=model,
      args=args_stage1,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer_stage1.train()

  # ==========================================
  # STAGE 2: FULL FINE-TUNING
  # ==========================================
  print("\n" + "="*50)
  print("🔥 STAGE 2: Full Fine-Tuning")
  print("="*50)
  model = unfreeze_backbone(model)

  args_stage2 = TrainingArguments(
      output_dir="custom-guardrail/outputs/checkpoints/stage2",
      learning_rate=cfg.TRAIN_HP["full"]["learning_rate"],
      per_device_train_batch_size=batch_size,
      per_device_eval_batch_size=batch_size,
      num_train_epochs=cfg.TRAIN_HP["full"]["epochs"],
      eval_strategy="epoch",
      save_strategy="epoch",
      fp16=cfg.TRAIN_HP["full"]["fp16"],
      load_best_model_at_end=True,
      report_to="none"
  )

  # Initialize a fresh Trainer for Stage 2 to avoid AcceleratorState conflicts
  trainer_stage2 = Trainer(
      model=model,
      args=args_stage2,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer_stage2.train()

  # Save final model payload
  final_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/final_model"))
  print(f"\n💾 Saving final production model to {final_dir}")
  trainer_stage2.save_model(final_dir)
  tokenizer.save_pretrained(final_dir)

if __name__ == "__main__":
  run_pipeline()

Overwriting custom-guardrail/src/finetuning/domain_finetuning/main.py


In [ ]:
!pip install -U transformers accelerate datasets evaluate scikit-learn -q

In [ ]:
!python custom-guardrail/src/finetuning/domain_finetuning/main.py

Loading tokenizer and model: microsoft/deberta-v3-small...
Loading weights: 100% 102/102 [00:00<00:00, 13910.55it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
clas

In [ ]:
%%writefile custom-guardrail/src/model_evaluation/gold_evaluation/core_logic_ge.py
import os
import sys
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# Setup relative paths
# Up 3 levels to custom-guardrail, then into configs
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../configs")))
# Up 2 levels to src, then into finetuning/domain_data (FIXED)
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../finetuning/domain_data")))

import pipeline_config as cfg
from core_logic_gd import get_tokenized_datasets

def evaluate_gold_set():
    print("🥇 Starting Gold Set Evaluation...")

    # 1. Load the fine-tuned model and the gold dataset
    model_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/final_model"))
    gold_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/gold.jsonl"))

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # We pass gold_path for both train/val arguments just to reuse our data loader, but we only use the second output
    _, gold_ds = get_tokenized_datasets(tokenizer, gold_path, gold_path, cfg.MAX_LENGTH)

    # 2. Run inference over the entire Gold Set
    trainer = Trainer(model=model)
    predictions, labels, _ = trainer.predict(gold_ds)

    # Convert raw logits to binary predictions (0 or 1)
    preds = np.argmax(predictions, axis=-1)

    # 3. Compute Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)

    # Confusion matrix returns True Negatives, False Positives, False Negatives, True Positives
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    # 4. Display Results
    print("\n" + "="*45)
    print("📊 GOLD SET EVALUATION RESULTS")
    print("="*45)
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"FPR:       {fpr:.2%}")
    print("="*45)

    # 5. Save the report to outputs
    report = {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": acc,
        "fpr": fpr,
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }

    report_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../../../outputs/pii/comparison_report.json"))
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=4)

    print(f"📄 Full JSON report saved to {report_path}")

if __name__ == "__main__":
    evaluate_gold_set()

Overwriting custom-guardrail/src/model_evaluation/gold_evaluation/core_logic_ge.py


In [ ]:
!python custom-guardrail/src/model_evaluation/gold_evaluation/core_logic_ge.py

🥇 Starting Gold Set Evaluation...
Loading weights: 100% 106/106 [00:00<00:00, 4713.80it/s]
Loading datasets from /content/outputs/pii/gold.jsonl...
Tokenizing datasets...
Map: 100% 7/7 [00:00<00:00, 1048.31 examples/s]
Map: 100% 7/7 [00:00<00:00, 1390.42 examples/s]
100% 1/1 [00:00<00:00, 2085.68it/s]

📊 GOLD SET EVALUATION RESULTS
F1 Score:  0.0000
Precision: 0.0000
Recall:    0.0000
Accuracy:  1.0000
FPR:       0.00%
📄 Full JSON report saved to /content/outputs/pii/comparison_report.json
